# ಪಾಠ 13 - ಏಜೆಂಟ್ ನೆನಪು


## ಸೆಟಪ್

ಈ ನೋಟ್ಬುಕ್ **Microsoft Agent Framework** (MAF) ಬಳಸಿ **ಸ್ಥಾಯೀ ಸ್ಮರಣೆ** ಸಹಿತ ಪ್ರಯಾಣ ಬುಕಿಂಗ್ ಏಜಂಟ್ ಹೇಗೆ ನಿರ್ಮಿಸುವುದನ್ನು ತೋರಿಸುತ್ತದೆ.

ನೀವು ಏಜಂಟ್ ಸ್ಮರಣೆಯ ವಿವಿಧ ಪ್ರಕಾರಗಳು — ಕೆಲಸ ಮಾಡುತ್ತಿರುವ, ಕಡಿಮೆ ಕಾಲದ, ಮತ್ತು ದೀರ್ಘಕಾಲದ — ಸಂವಾದಗಳ ನಡುವಿನ ಮಾಹಿತಿಯನ್ನು ಏಜಂಟ್ ಹೇಗೆ ಉಳಿಸಿಕೊಂಡು ಬಳಸುತ್ತವೆಯೋ ಆ ಬಗ್ಗೆ ತಿಳಿಯಿರಿ.

**ಆವಶ್ಯಕತೆಗಳು:**
- ಒಂದು Azure AI Foundry ಪ್ರಾಜೆಕ್ಟ್ ಜೊತೆಗೆ ನಿಯೋಜಿಸಲಾದ ಚಾಟ್ ಮಾದರಿ (ಉದಾ: `gpt-4o-mini`).
- Azure CLI ಮೂಲಕ ಲಾಗಿನ್ ಆಗಿರುವುದು — ನಿಮ್ಮ ಟರ್ಮಿನಲ್‌ನಲ್ಲಿ `az login` ಅನ್ನು 실행ಿಸಿ.
- `AZURE_AI_PROJECT_ENDPOINT` — ನಿಮ್ಮ Azure AI Foundry ಪ್ರಾಜೆಕ್ಟ್ ಎಂಡ್‌ಪಾಯಿಂಟ್.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ನಿಯೋಜಿಸಲಾದ ಮಾದರಿಯ ಹೆಸರು.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ಏಜೆಂಟ್ ಮೆಮೊರಿಯ ಪ್ರಕಾರಗಳು

AI ಏಜೆಂಟ್‌ಗಳು ವಿಭಿನ್ನ ರೀತಿಯ ಮೆಮೊರಿ ಬಳಸಬಹುದು, ಪ್ರತಿ ಒಂದು ವಿಭಿನ್ನ ಉದ್ದೇಶಕ್ಕೆ ಸೇವೆ ನೀಡುತ್ತದೆ:

### ಕೆಲಸ ಮಾಡುವ ಮೆಮೊರಿ
ಸಂವಹನ ಮೇಲುಗುಟ್ಟು — ಒಂದು ಸೆಷನ್‌ನಲ್ಲಿ ವಿನಿಮಯವಾಗುವ ಸಂದೇಶಗಳು. ಏಜೆಂಟ್ coherence ಕಾಪಾಡಿಕೊಳ್ಳಲು ಇದೇ ಮೇಲುಗುಟ್ಟಿನ ಹಳೆಯ ಸಂದೇಶಗಳಿಗೆ ಹಿಂತಿರುಗಿ ನೋಡಬಹುದು. MAF ನಲ್ಲಿ ಇದನ್ನು **`agent.create_session()`** ಮೂಲಕ ಸೃಷ್ಟಿಸಲಾಗುತ್ತದೆ, ಇದು `AgentSession` ಅನ್ನು ಹಿಂತಿರುಗಿಸುತ್ತದೆ.

### ಸಂಕ್ಷಿಪ್ತಾವಧಿ ಮೆಮೊರಿ
ಒಂದು ಕಾರ್ಯ ಅಥವಾ ಸೆಷನ್ ಅವಧಿಗಾಗಿಯೇ ಉಳಿಯುವ ಮಾಹಿತಿ ಆದರೆ ಶಾಶ್ವತವಾಗಿ ಸಂಗ್ರಹಿಸಲಾಗದು. ಉದಾಹರಣೆಗೆ, ಏಜೆಂಟ್ ಬಹು-ತಿರುವಿನ ಯೋಜನಾ ಸಂಭಾಷಣೆಯ ಸಮಯದಲ್ಲಿ ವಿಚಾರಗಳನ್ನು ಸಂಗ್ರಹಿಸಿ ಅಂತಿಮ ಸಂಚಿ ತಯಾರಿಸಲು ಬಳಸಬಹುದು.

### ದೀರ್ಘಕಾಲಿಕ ಮೆಮೊರಿ
ಸೆಷನ್‌ಗಳನ್ನು **ಅಧಿಕವಾಗಿ** ಉಳಿಸುವ ಅಭಿರುಚಿಗಳು ಮತ್ತು ವಿಚಾರಗಳು. ಹಿಂದಿರುಗಿ ಬರುವ ಬಳಕೆದಾರರು ತಮ್ಮ ಆಹಾರ ನಿರ್ಬಂಧಗಳು ಅಥವಾ ಪ್ರಯಾಣ ಶೈಲಿಯನ್ನು ಪುನರಾವರ್ತಿಸಲು ನಡೆಯಬಾರದು. ದೀರ್ಘಕಾಲಿಕ ಮೆಮೊರಿ ಸಾಮಾನ್ಯವಾಗಿ ಬಾಹ್ಯ ಸಂಗ್ರಹಣೆ—ಡೇಟಾಬೇಸ್, ಫೈಲ್ ಅಥವಾ ವೆಕ್ಟರ್ ಸೂಚ್ಯಂಕ—ಮೂಲಕ ಬೆಂಬಲಿತವಾಗಿದ್ದು, ಟೂಲುಗಳ ಮೂಲಕ ಏಜೆಂಟ್ ಗೆ ಗಣಿಸಲಾಗುತ್ತದೆ.


## ಸೆನ್ಸಸ್‌ಗಳೊಂದಿಗೆ ಕಾರ್ಯಚಟುವಟಿಕೆ ಸ್ಮೃತಿ

ಎಳಗಿನ ಸ್ಮೃತಿ ಸ್ವರೂಪವೇ ಸಂಭಾಷಣೆ ಸೆನ್ಸಸ್ ಆಗಿದೆ. ನೀವು ಒಂದೇ ಸೆನ್ಸಸ್ ಅನ್ನು ( `agent.create_session()` ಮೂಲಕ ರಚಿಸಿದ)連續 `agent.run()` ಕರೆಗಳಿಗೆ ಪಾಸ್ ಮಾಡಿದಾಗ, ಏಜೆಂಟ್ ಆ ಸಂಭಾಷಣೆಯ ಸಂಪೂರ್ಣ ಇತಿಹಾಸವನ್ನು ನೋಡುತ್ತದೆ ಮತ್ತು ಹಿಂದಿನ ವಿವರಗಳನ್ನು ಸ್ಮರಿಸಬಹುದು.

ನಾವು ಒಂದು ಪ್ರಯಾಣ ಏಜೆಂಟ್ ರಚಿಸಲಿದ್ದೇವೆ ಮತ್ತು ಕಾರ್ಯಚಟುವಟಿಕೆ ಸ್ಮೃತಿಯನ್ನು ಪ್ರದರ್ಶಿಸೋಣ.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ಕಾರ್ಯನಿರ್ವಹಣೆಯ ಬಗ್ಗೆ ಜಾಗೃತಿಗಳು ಬಜೆಟ್ ಅನ್ನು ಸರಿಯಾಗಿ ನೆನಪಿಟ್ಟುಕೊಂಡಿವೆ ಏಕೆಂದರೆ ಇಬ್ಬರು ಸಂದೇಶಗಳು ಒಂದೇ ಸೆಷನ್ ಹಂಚಿಕೊಳ್ಳುತ್ತವೆ. ಇದನ್ನು **ಕಾರ್ಯಸ್ಮೃತಿ** ಎಂದು ಕರೆಯಲಾಗುತ್ತದೆ — ಇದು ಸೆಷನ್ ಬದುಕಿನಾವಧಿಗೆ ಮಾತ್ರ ಇರುತ್ತದೆ.

### ಹೊಸ ಥ್ರೆಡ್‌ನೊಂದಿಗೆ ಏನು ಘಟುತ್ತದೆ?

ನಾವು **ಹೊಸ** ಸೆಷನ್ ಸೃಷ್ಟಿಸಿದರೆ, ಕಾರ್ಯನಿರ್ವಹಣೆಗೆ ಹಿಂದಿನ ಸಂಭಾಷಣೆ ಕುರಿತು ಯಾವುದೇ ಸ್ಮೃತಿ ಇಲ್ಲ:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## ದೀರ್ಘಕಾಲಿಕ ಸ್ಮೃತಿ ಮಾದರಿ

ಬಳಕೆದಾರರ ಮೆಚ್ಚುಗೆಗಳನ್ನು **ಸೆಷನ್‌ಗಳ ಆಸ್ಪತ್ರೆಗೆ** ನೆನಪಿಡಲು, ಸಂಭಾಷಣೆ ಯಾರು ಬಾಹ್ಯದಲ್ಲಿ ಇರುವ ಸ್ಥಿರ ಸಂಗ್ರಹಣೆಯನ್ನು ಅಗತ್ಯವಿದೆ. ಏಜೆಂಟ್ ಈ ಸಂಗ್ರಹಣೆಯನ್ನು **ಉಪಕರಣಗಳ ಮೂಲಕ** - ಉಳಿಸುವ ಮತ್ತು ಮಾಹಿತಿ ಪಡೆಯಲು ಕರೆಮಾಡಬಹುದಾದ ಕಾರ್ಯಗಳ ಮೂಲಕ ಪ್ರಾಪ್ತಿಗೊಳಿಸುತ್ತದೆ.

ಕೆಳಗೆ ನಾವು ಸರಳ ಇನ್ಮೆಮೊರಿ ಮೆಚ್ಚುಗೆಯ ಸಂಗ್ರಹಣೆಯನ್ನು ಅನುಷ್ಟಾನ ಮಾಡುತ್ತೇವೆ (ಉತ್ಪಾದನೆಯಲ್ಲಿ ಇದನ್ನು ಡೇಟಾಬೇಸ್ ಅಥವಾ ವೆಕ್ಟರ್ ಸೂಚಕದೊಂದಿಗೆ ಬ್ಯಾಕ್ ಮಾಡುತ್ತಾರೆ) ಮತ್ತು ಅದನ್ನು ಏಜೆಂಟ್ ಉಪಯೋಗಿಸಬಹುದಾದ ಉಪಕರಣಗಳಾಗಿ ಬಹಿರಂಗಪಡಿಸುತ್ತೇವೆ.

### ವಾಸ್ತುಶಿಲ್ಪ  
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### ಘಟನೆ 1 — ಮೊದಲ ಬಾರಿ ಬಳಕೆದಾರರು ವಾರ್ಷಿಕೋತ್ಸವ ಪ್ರಯಾಣವನ್ನು ಬುಕ್ ಮಾಡುತ್ತಾರೆ

ಸಾರಾಹ್ ಮೊದಲ ಬಾರಿ ಭೇಟಿ ನೀಡುತ್ತಾರೆ. ಏಜೆಂಟ್ ಅವಳ ಇಚ್ಛೆಗಳನ್ನು ಉಪಕರಣಗಳ ಮೂಲಕ ಸಂಗ್ರಹಿಸಿ ಅವುಗಳನ್ನು ಆಸರೆಯಾಗಿ ಹೋಟೆಲ್ಗಳನ್ನು ಶಿಫಾರಸು ಮಾಡಬೇಕಾಗಿದೆ.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### ಘಟನಾಕ್ರಮ 2 — ಸಾರೆ ಹಬ್ಬಿಸಿದ ಕೆಲ ವಾರಗಳ ಬಳಿಕ ಮರಳುತ್ತಾರೆ

ಸಾರೆ **ಹೊಸ ತಂತಿಯ** ಪ್ರಾರಂಭ ಮಾಡುತ್ತಾರೆ (ಹೊಸ ಸೆಶನ್ ಅನ್ನು ಅನುಕರಿಸುತ್ತಿದ್ದಾರೆ). ವರ್ಕಿಂಗ್ ಮೆಮೊರಿ ಖಾಲಿ ಇದೆ, ಆದರೆ ದೀರ್ಘಕಾಲಿಕ ಪ್ರಾಧಾನ್ಯತೆಯ ಸಂಗ್ರಹದಲ್ಲಿ ಇನ್ನೂ ಅವರ ಮಾಹಿತಿ ಇದೆ. ಏಜೆಂಟ್ ಅದನ್ನು ಮರುಪಡೆದು ವೈಯಕ್ತಿಕ ಶಿಫಾರಸುಗಳಿಗೆ ಬಳಸಬೇಕು.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಮೂರು ವಿಧದ ಏಜೆಂಟ್ ಮೆಮೊರಿ ಮತ್ತು ಅದನ್ನು Microsoft Agent Framework ಬಳಿಸಿ ಹೇಗೆ ಅನುಷ್ಟಾನ ಮಾಡುವುದು ಎಂದು ಕಲಿತಿರಿ:

| ಮೆಮೊರಿ ವಿಧ | MAF ಯಂತ್ರಾಚರಣೆ | ಜೀವಾವಧಿ |
|---|---|---|
| **ಕಾರ್ಯಾಚರಣೆ** | `agent.create_session()` | ಏಕ ಸಂಭಾಷಣೆ |
| **ರೂಪಾಂತರಿ-ಕಾಲಿಕ** | ತುಣುಕು ಒಳಗಿನ ಸಂಗ್ರಹಿತ ಸಾಂದರ್ಭಿಕತೆ | ಏಕ ಕಾರ್ಯ / ಸೆಷನ್ |
| **ದೀರ್ಘಕಾಲಿಕ** | `@tool` ಕಾರ್ಯಗಳ ಮೂಲಕ ಪ್ರವೇಶಿಸುವ ಬಾಹ್ಯ ಸಂಗ್ರಹಣೆ | ಸೆಷನ್ ಗಳ ನಡುವೆ |

### ಪ್ರಮುಖ ಅಂಶಗಳು
1. **`agent.create_session()`** ಕಾರ್ಯಾಚರಣೆ ಮೆಮೊರಿಯನ್ನು ಒದಗಿಸುತ್ತದೆ — ಏಜೆಂಟ್ ಒಂದು ಸೆಷನ್ ಒಳಗಿನ ಸಂಪೂರ್ಣ ಸಂಭಾಷಣೆ ಇತಿಹಾಸವನ್ನು ನೋಡಬಹುದು.
2. **ಹೊಸ ಸೆಷನ್‌ಗಳು ಸಾಂದರ್ಭಿಕತೆಯನ್ನು ಕಳೆದುಕೊಂಡಿವೆ** — ದೀರ್ಘಕಾಲಿಕ ಮೆಮೊರಿ ಇಲ್ಲದೆ ಏಜೆಂಟ್ ಕಳೆದ ಸಂಭಾಷಣೆಗಳನ್ನು ನೆನಪಿಟ್ಟುಕೊಳ್ಳಲು ಸಾಧ್ಯವಿಲ್ಲ.
3. **`@tool` ಕಾರ್ಯಗಳು** ಸಂಪರ್ಕವನ್ನು ಭೇಟಿ ಮಾಡುತ್ತವೆ — ಅವು ಏಜೆಂಟಿಗೆ ಸ್ಥಿರ ಸಂಗ್ರಹಣೆಯಿಂದ ಮಾಹಿತಿ ಉಳಿಸುವ ಮತ್ತು ಪಡೆಯಲು ಅನುಮತಿಸುತ್ತವೆ.
4. **ವ್ಯಕ್ತಿಗತೀಕರಣ ಸಮಯದೊಂದಿಗೆ ಸುಧಾರಿಸುತ್ತದೆ** — ಹೆಚ್ಚಿನ ಆಯ್ಕೆಗಳು ಸಂಗ್ರಹಿಸಿದಂತೆ, ಏಜೆಂಟ್‌ನ ಶಿಫಾರಸುಗಳು ಉತ್ತಮವಾಗುತ್ತವೆ.

### ನೈಜ ಜಗತ್ತಿನ ಅನ್ವಯಗಳು
- **ಗ್ರಾಹಕ ಸೇವೆ**: ಗ್ರಾಹಕ ಇತಿಹಾಸ ಮತ್ತು ಆಯ್ಕೆಯನ್ನು ನೆನಪಿಡಿ
- **ವೈಯಕ್ತಿಕ ಸಹಾಯಕರು**: ದಿನಗಳು ಅಥವಾ ವಾರಗಳ ನಡುವೆ ಸಾಂದರ್ಭಿಕತೆ ಕಾಪಾಡು
- **ಆರೋಗ್ಯಸೇವೆ**: ರೋಗಿಯ ಮಾಹಿತಿಗಳು ಮತ್ತು ಆಯ್ಕೆಗಳು ಟ್ರ್ಯಾಕ್ ಮಾಡು
- **ಇ-ಕಾಮರ್ಸ್**: ಇತಿಹಾಸ ಆಧರದ ವೈಯಕ್ತಿಕ ಶಾಪಿಂಗ್

### ಮುಂದಿನ ಹಾದಿಗಳು
- ನೆನಪು ಸಂಖ್ಯೆ ಭಂಡಾರ ಅಥವಾ ವೆಕ್ಟರ್ ಸ್ಟೋರ್ ( ಉದಾ. Azure AI Search) ಮೂಲಕ ಬದಲಾಯಿಸಿ
- ಸಮಯ-ಸಂಚಿತ ಮಾಹಿತಿಗಾಗಿ ಮೆಮೊರಿ ಅವಧಿ ಸೇರಿಸಿ
- ಹಂಚಿಕೊಂಡ ಮೆಮೊರಿ ಹೊಂದಿರುವ ಬಹು-ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳು ನಿರ್ಮಿಸಿ
- ಜ್ಞಾನ-ಗ್ರಾಫ್ ಆಧಾರಿತ ಮೆಮೊರಿಗಾಗಿ Cognee ನೋಟುಬುಕ್ ಅನ್ವೇಷಿಸಿ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ನಿಬಂಧನಾ**:
ಈ ದಾಖಲೆಯನ್ನು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಿಖರತೆಯ ಬಗ್ಗೆ ನಾವು ಪ್ರಯತ್ನಿಸಿದರೂ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸತ್ಯತೆಗಳು ಇರಬಹುದು ಎಂದು ತಿಳಿಯಿರಿ. ಮೂಲ ಭಾಷೆಯಲ್ಲಿ ಇರುವ ಮೂಲ ದಾಖಲೆ ಆಡಳಿತಾತ್ಮಕ ಮೂಲವಾಗಿ ಪರಿಗಣಿಸಬೇಕಾಗುತ್ತದೆ. ಗಂಭೀರ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡುತ್ತೇವೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವುದರಿಂದ ಉದ್ಭವಿಸಿದ ಯಾವುದೇ ತಪ್ಪು ಗ್ರಹಿಕೆಗಳ ಅಥವಾ ತಪ್ಪಾಗಿ ಅರ್ಥಮಾಡಿಕೊಳ್ಳುವಿಕೆಯjimaಗಾಗಿ ನಾವು ಜವಾಬ್ದಾರಿಯಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
